# 🗡️ YOLOv26-Based Kris Detection & Classification
## Madura Cultural Heritage Preservation — Google Colab Training Pipeline

Notebook ini melatih dan menguji model **YOLOv26** menggunakan dataset anotasi keris yang dihasilkan dari Dashboard Web (dataset_keris).

> **Catatan:** Class label dibaca **secara dinamis** dari folder dataset — tidak perlu diubah manual saat class bertambah.

---
### Alur Kerja:
1. ⚙️ Setup Environment
2. 📦 Upload / Mount Dataset
3. 🔍 Scan Dataset & Bangun Class Map Dinamis
4. ✂️ Split Train/Val
5. 📝 Generate dataset_keris.yaml
6. 🏋️ Training YOLOv26
7. 📊 Evaluasi & Visualisasi
8. 📷 Webcam Inference Realtime

## 1. ⚙️ Setup Environment

In [ ]:
# Verifikasi GPU T4/A100 aktif
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Instal semua dependensi yang diperlukan
!pip install -q 'ultralytics>=8.3.50' beautifulsoup4 pillow requests pandas tqdm opencv-python matplotlib PyYAML
print("✅ Instalasi selesai.")

## 2. 📦 Upload Dataset

Dataset yang dihasilkan dari Dashboard Web memiliki struktur:
```
dataset_keris/
├── images/
│   ├── keris_luk_3/        ← folder per kelas
│   ├── keris_luk_5/
│   ├── keris_lurus/
│   ├── keris_carang_soka_/
│   └── ...                 ← 27 folder kelas
├── labels/
│   ├── keris_carang_soka_/  ← hanya folder yang sudah dianotasi
│   └── ...
└── metadata/
    ├── annotations_db.json
    └── checkpoint.json
```

**Pilih salah satu opsi upload:**

In [ ]:
# ==========================================
# OPSI A: Upload dari Google Drive
# ==========================================
# Zip dataset_keris lalu upload ke Google Drive terlebih dahulu
# Jalankan cell ini jika menggunakan Google Drive

from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path zip di Drive Anda:
ZIP_PATH = '/content/drive/MyDrive/dataset_keris.zip'

import os
if os.path.exists(ZIP_PATH):
    !unzip -q "{ZIP_PATH}" -d /content/
    print("✅ Dataset berhasil diekstrak dari Google Drive.")
else:
    print(f"⚠️ File tidak ditemukan: {ZIP_PATH}")
    print("   Gunakan Opsi B (upload manual) sebagai alternatif.")

In [ ]:
# ==========================================
# OPSI B: Upload Manual via Browser
# ==========================================
# Jalankan cell ini untuk upload file zip dataset_keris langsung dari komputer

from google.colab import files
import os

print("Silakan pilih file dataset_keris.zip dari komputer Anda...")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        !unzip -q "{filename}" -d /content/
        print(f"✅ Dataset diekstrak dari: {filename}")
    else:
        print(f"⚠️ File bukan .zip: {filename}")

## 3. 🔍 Scan Dataset & Bangun Class Map Dinamis

Cell ini membaca label secara otomatis dari:
- **Folder `images/`**: mendaftarkan semua kelas yang ada
- **File `annotations_db.json`**: membaca label YOLO yang benar-benar dipakai di anotasi

Sehingga class map selalu sinkron dengan kondisi dataset nyata.

In [ ]:
import os
import json
import shutil
import random
import glob
from collections import Counter, defaultdict
from pathlib import Path

# ── Path konfigurasi ──────────────────────────────────────────────────────────
DATASET_ROOT  = Path('/content/dataset_keris')
IMAGES_DIR    = DATASET_ROOT / 'images'
LABELS_DIR    = DATASET_ROOT / 'labels'
METADATA_DIR  = DATASET_ROOT / 'metadata'
ANNOTATIONS_DB = METADATA_DIR / 'annotations_db.json'

assert DATASET_ROOT.exists(), (
    f"❌ Folder dataset tidak ditemukan: {DATASET_ROOT}\n"
    "   Pastikan Anda sudah mengekstrak dataset_keris.zip ke /content/"
)

# ── 1. Kumpulkan semua label YOLO yang CONFIRMED dari annotations_db ──────────
yolo_labels_used = set()
annotated_images = {}  # rel_path → {status, boxes, label_file_exists}

if ANNOTATIONS_DB.exists():
    with open(ANNOTATIONS_DB, 'r', encoding='utf-8') as f:
        db = json.load(f)
    
    for rel_path, data in db.items():
        if data.get('status') != 'done':
            continue
        confirmed_labels = [
            b['label'] for b in data.get('boxes', [])
            if b.get('confirmed') and b.get('label')
        ]
        if confirmed_labels:
            yolo_labels_used.update(confirmed_labels)
            annotated_images[rel_path] = {
                'boxes': data['boxes'],
                'confirmed_labels': confirmed_labels
            }
    print(f"✅ Membaca annotations_db.json — {len(db)} entri, {len(annotated_images)} gambar teranotasi done.")
else:
    print("⚠️ annotations_db.json tidak ditemukan — hanya membaca dari folder images.")
    db = {}

# ── 2. Scan folder images untuk semua kelas yang ada ─────────────────────────
all_image_classes = sorted([
    d.name for d in IMAGES_DIR.iterdir()
    if d.is_dir() and d.name not in ('train', 'val')
])

print(f"\n📁 Ditemukan {len(all_image_classes)} folder kelas di images/:")
class_image_counts = {}
for cls in all_image_classes:
    imgs = list((IMAGES_DIR / cls).glob('*.*'))
    imgs = [f for f in imgs if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    class_image_counts[cls] = len(imgs)
    print(f"   {cls:<35} {len(imgs):>4} gambar")

total_images = sum(class_image_counts.values())
print(f"\n📊 Total gambar: {total_images}")

# ── 3. Scan folder labels untuk class yang sudah punya label YOLO .txt ────────
labeled_classes = sorted([
    d.name for d in LABELS_DIR.iterdir()
    if d.is_dir() and d.name not in ('train', 'val')
]) if LABELS_DIR.exists() else []

print(f"\n🏷️  Ditemukan {len(labeled_classes)} folder kelas di labels/:")
class_label_counts = {}
for cls in labeled_classes:
    txts = list((LABELS_DIR / cls).glob('*.txt'))
    class_label_counts[cls] = len(txts)
    print(f"   {cls:<35} {len(txts):>4} label .txt")

# ── 4. Bangun CLASS MAP dinamis dari label yang benar-benar dipakai ───────────
# Prioritaskan label YOLO dari confirmed boxes di annotations_db
# Urutkan: keris_lurus, keris_luk_* (ascending), lalu dhapur/unknown

def label_sort_key(name):
    if name == 'keris_lurus':      return (0, 0, name)
    if name.startswith('keris_luk_'):
        try:
            return (1, int(name.split('_')[-1]), name)
        except:
            return (1, 999, name)
    if name == 'keris_unknown':    return (3, 0, name)
    return (2, 0, name)  # dhapur-based

if yolo_labels_used:
    # Gunakan label dari annotations_db (lebih akurat)
    sorted_labels = sorted(yolo_labels_used, key=label_sort_key)
    label_source = "annotations_db.json (confirmed boxes)"
else:
    # Fallback: gunakan nama folder images sebagai class name
    sorted_labels = sorted(all_image_classes, key=label_sort_key)
    label_source = "nama folder images/ (fallback)"

CLASS_MAP = {label: idx for idx, label in enumerate(sorted_labels)}
CLASS_NAMES = sorted_labels
NC = len(CLASS_NAMES)

print(f"\n🗺️  Class map dibangun dari: {label_source}")
print(f"   Jumlah kelas (nc): {NC}")
print("   ID → Label:")
for idx, label in enumerate(CLASS_NAMES):
    print(f"     {idx:>2}: {label}")

## 4. ✂️ Split Dataset Train/Val

Membagi dataset per kelas dengan rasio **80% train / 20% val**.
Hanya gambar yang **memiliki file label `.txt`** yang dimasukkan ke split (gambar tanpa label diabaikan untuk training, agar tidak error).

In [ ]:
import random
import shutil
from pathlib import Path

TRAIN_RATIO = 0.8
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Buat folder train/val
for split in ['train', 'val']:
    (IMAGES_DIR / split).mkdir(exist_ok=True)
    (LABELS_DIR / split).mkdir(exist_ok=True)

train_count = 0
val_count   = 0
skipped     = 0

print("🔀 Memulai split dataset...\n")

for cls in all_image_classes:
    cls_img_dir   = IMAGES_DIR / cls
    cls_label_dir = LABELS_DIR / cls
    
    # Kumpulkan gambar yang punya label .txt yang valid
    paired = []
    for img_file in cls_img_dir.iterdir():
        if img_file.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        label_file = cls_label_dir / (img_file.stem + '.txt')
        if label_file.exists() and label_file.stat().st_size > 0:
            paired.append((img_file, label_file))
        else:
            skipped += 1
    
    if not paired:
        print(f"   ⚠️  {cls}: tidak ada pasangan image+label — dilewati")
        continue
    
    random.shuffle(paired)
    split_idx   = max(1, int(len(paired) * TRAIN_RATIO))
    train_files = paired[:split_idx]
    val_files   = paired[split_idx:]
    # Minimal 1 gambar untuk val jika hanya ada 1 gambar total
    if not val_files and len(paired) > 1:
        val_files   = paired[-1:]
        train_files = paired[:-1]
    
    for img_f, lbl_f in train_files:
        shutil.copy2(img_f, IMAGES_DIR / 'train' / img_f.name)
        shutil.copy2(lbl_f, LABELS_DIR / 'train' / lbl_f.name)
        train_count += 1
    
    for img_f, lbl_f in val_files:
        shutil.copy2(img_f, IMAGES_DIR / 'val' / img_f.name)
        shutil.copy2(lbl_f, LABELS_DIR / 'val' / lbl_f.name)
        val_count += 1
    
    print(f"   ✅ {cls:<35} → train: {len(train_files):>3}  val: {len(val_files):>3}")

print(f"\n{'='*55}")
print(f"   Total train : {train_count} gambar")
print(f"   Total val   : {val_count} gambar")
print(f"   Dilewati    : {skipped} gambar (tidak ada label .txt)")
print(f"{'='*55}")

if train_count == 0:
    print("\n❌ Tidak ada gambar berhasil di-split!")
    print("   Pastikan folder labels/ berisi file .txt yang valid.")
    print("   Jalankan anotasi di Dashboard Web terlebih dahulu.")
else:
    print("\n✅ Split selesai — siap untuk training!")

## 5. 📝 Generate `dataset_keris.yaml`

File YAML dibuat secara **dinamis** berdasarkan class map yang telah dibangun.

In [ ]:
import yaml

YAML_PATH = Path('/content/dataset_keris.yaml')

# Cek apakah folder val kosong (fallback ke train)
val_imgs = list((IMAGES_DIR / 'val').glob('*.*')) if (IMAGES_DIR / 'val').exists() else []
val_path = 'images/val' if val_imgs else 'images/train'

dataset_config = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': val_path,
    'nc': NC,
    'names': CLASS_NAMES
}

with open(YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True)

# Tampilkan isi YAML
print(f"✅ dataset_keris.yaml berhasil dibuat: {YAML_PATH}")
print(f"   val path: {val_path} {'(fallback ke train — dataset val kosong)' if val_path == 'images/train' else ''}")
print("\n📄 Isi file YAML:")
print("-" * 50)
with open(YAML_PATH, 'r') as f:
    print(f.read())

## 6. 🏋️ Training YOLOv26

Model dilatih menggunakan GPU T4/A100 yang tersedia di Google Colab.

| Parameter | Nilai | Keterangan |
|---|---|---|
| `epochs` | 50 | Jumlah iterasi penuh (naikkan ke 100+ untuk dataset besar) |
| `imgsz` | 640 | Resolusi input gambar (standar YOLO) |
| `batch` | 16 | Batch size (turunkan ke 8 jika OOM) |
| `device` | 0 | GPU index (0 = GPU pertama) |

In [ ]:
# Verifikasi sebelum training
train_imgs = list((IMAGES_DIR / 'train').glob('*.*'))
train_lbls = list((LABELS_DIR / 'train').glob('*.txt'))

print("🔍 Verifikasi pra-training:")
print(f"   Gambar train  : {len(train_imgs)}")
print(f"   Label train   : {len(train_lbls)}")
print(f"   YAML path     : {YAML_PATH}")
print(f"   Jumlah kelas  : {NC}")
print(f"   Class list    : {CLASS_NAMES}")

if len(train_imgs) == 0:
    print("\n❌ Tidak ada gambar di folder train! Training dibatalkan.")
    print("   Periksa langkah upload dan split dataset.")
elif len(train_lbls) == 0:
    print("\n❌ Tidak ada label .txt di folder train! Training dibatalkan.")
else:
    print("\n✅ Semua siap untuk training!")

In [ ]:
from ultralytics import YOLO

# ── Konfigurasi Training ──────────────────────────────────────────────────────
EPOCHS    = 50      # Naikkan ke 100-150 untuk hasil lebih baik
IMG_SIZE  = 640
BATCH     = 16      # Turunkan ke 8 jika mendapat error CUDA out of memory
MODEL_PT  = 'yolo26n.pt'  # Pretrained nano — paling cepat untuk fine-tune
PROJECT   = '/content/runs/keris'
RUN_NAME  = 'yolov26_madura_kris'

# Muat model
model = YOLO(MODEL_PT)
print(f"✅ Model dimuat: {MODEL_PT}")
print(f"   Arsitektur: {model.info(verbose=False)}")

# Mulai training
results = model.train(
    data    = str(YAML_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = PROJECT,
    name    = RUN_NAME,
    exist_ok= True,
    patience= 20,      # Early stopping jika tidak ada improvement
    save    = True,
    plots   = True,
    verbose = True,
)

print("\n🎉 Training selesai!")
BEST_MODEL = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
LAST_MODEL = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'
print(f"   Model terbaik : {BEST_MODEL}")
print(f"   Model terakhir: {LAST_MODEL}")

## 7. 📊 Evaluasi & Visualisasi Hasil Training

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

RUN_DIR = Path(PROJECT) / RUN_NAME

# Tampilkan grafik training
plot_files = [
    ('results.png', 'Training Metrics'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('val_batch0_pred.jpg', 'Val Batch Predictions'),
]

for fname, title in plot_files:
    fpath = RUN_DIR / fname
    if fpath.exists():
        fig, ax = plt.subplots(figsize=(14, 8))
        img = mpimg.imread(str(fpath))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print(f"   ℹ️  {fname} belum tersedia")

# Validasi metrik pada val set
print("\n📈 Validasi model terbaik pada val set...")
if BEST_MODEL.exists():
    model_best = YOLO(str(BEST_MODEL))
    metrics = model_best.val(
        data   = str(YAML_PATH),
        imgsz  = IMG_SIZE,
        device = 0 if torch.cuda.is_available() else 'cpu',
        verbose= True
    )
    print(f"\n📊 Hasil Validasi:")
    print(f"   mAP@0.5     : {metrics.box.map50:.4f}")
    print(f"   mAP@0.5:0.95: {metrics.box.map:.4f}")
    print(f"   Precision   : {metrics.box.mp:.4f}")
    print(f"   Recall      : {metrics.box.mr:.4f}")
else:
    print("   ⚠️ best.pt belum tersedia — jalankan training terlebih dahulu.")

In [ ]:
# Simpan model terbaik ke Google Drive
import shutil
from google.colab import drive

# Pastikan Drive sudah di-mount (jalankan cell upload drive jika belum)
DRIVE_SAVE_PATH = '/content/drive/MyDrive/yolov26_kris_best.pt'

if BEST_MODEL.exists():
    shutil.copy2(str(BEST_MODEL), DRIVE_SAVE_PATH)
    print(f"✅ Model tersimpan ke Google Drive: {DRIVE_SAVE_PATH}")
else:
    print("⚠️ best.pt tidak ditemukan — training mungkin belum selesai.")

## 8. 📷 Webcam Inference Realtime

Gunakan webcam browser untuk mendeteksi keris secara realtime dengan metadata budaya Madura.

In [ ]:
import json
from pathlib import Path

# Coba muat knowledge base dari berbagai kemungkinan lokasi
KB_CANDIDATES = [
    Path('/content/backend/knowledge_base.json'),
    Path('/content/knowledge_base.json'),
    DATASET_ROOT.parent / 'backend' / 'knowledge_base.json',
]

cultural_db = {}
for kb_path in KB_CANDIDATES:
    if kb_path.exists():
        with open(kb_path, 'r', encoding='utf-8') as f:
            cultural_db = json.load(f)
        print(f"✅ Knowledge base dimuat dari: {kb_path}")
        print(f"   Keys: {list(cultural_db.keys())}")
        break
else:
    print("⚠️ knowledge_base.json tidak ditemukan — metadata budaya tidak ditampilkan.")
    print("   Upload file backend/knowledge_base.json dari repositori ke Colab.")

# Peta luk dari nama label
LUK_MAKNA = {
    0:  "Lurus — Kejujuran, kepolosan, kesederhanaan",
    3:  "Luk 3 — Trimurti: keseimbangan penciptaan, pemeliharaan & peleburan",
    5:  "Luk 5 — Pancaindra, lambang kesempurnaan manusia",
    7:  "Luk 7 — Tujuh lapis langit, derajat spiritual tertinggi",
    9:  "Luk 9 — Wali Songo, kepemimpinan spiritual",
    11: "Luk 11 — Kesempurnaan kekuatan batin dan perlindungan",
    13: "Luk 13 — Kesakralan tertinggi, kekuatan magis raja/panglima",
    15: "Luk 15 — Paling langka, wahyu agung raja-dewa",
    17: "Luk 17 — Keagungan spiritual tertinggi",
    27: "Luk 27 — Luar biasa langka, pusaka pewayangan",
}

def extract_luk_from_label(label):
    """Ekstrak jumlah luk dari nama label YOLO."""
    if 'lurus' in label:
        return 0
    if 'luk_' in label:
        try:
            return int(label.split('luk_')[-1].rstrip('_'))
        except:
            pass
    return None

def get_cultural_info(label):
    """Ambil informasi budaya dari knowledge base berdasarkan label YOLO."""
    luk = extract_luk_from_label(label)
    luk_makna = LUK_MAKNA.get(luk, "Luk tidak diketahui")
    
    # Cari di knowledge base
    kb_luk = cultural_db.get('luk', {})
    luk_info = kb_luk.get(str(luk), {}).get('makna', luk_makna) if luk is not None else luk_makna
    
    # Contoh dapur berdasarkan luk
    dapur_default = {
        0: 'Tilam Upih / Brojol', 3: 'Ron Warih', 5: 'Carita',
        7: 'Carubuk', 9: 'Sabuk Inten', 11: 'Carang Soka',
        13: 'Sengkelat'
    }
    dapur = dapur_default.get(luk, label.replace('keris_', '').replace('_', ' ').title())
    
    tangguh = cultural_db.get('tangguh', {}).get('madura', {}).get('nama', 'Tangguh Madura')
    pamor   = list(cultural_db.get('pamor', {}).values())[0].get('nama', 'Beras Wutah') if cultural_db.get('pamor') else 'Beras Wutah'
    unesco  = cultural_db.get('status_budaya', {}).get('unesco', 'UNESCO ICH 2008')
    
    return {'luk': luk, 'luk_makna': luk_info, 'dapur': dapur, 'tangguh': tangguh, 'pamor': pamor, 'unesco': unesco}

print("✅ Fungsi cultural info siap.")

In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

def take_photo(filename='photo.jpg', quality=0.85):
    """Ambil gambar dari webcam browser melalui JavaScript Colab."""
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      div.style.cssText = 'padding:10px; background:#1a0e00; border-radius:8px; display:inline-block;';
      
      const label = document.createElement('div');
      label.innerHTML = '🗡️ <b style="color:#C9A84C;">YOLOv26 Kris Detector</b> — Posisikan keris di depan kamera';
      label.style.cssText = 'color:#f5e6c8; font-family:sans-serif; font-size:13px; margin-bottom:8px;';
      
      const video = document.createElement('video');
      video.style.cssText = 'display:block; border-radius:8px; border:2px solid #C9A84C;';
      video.setAttribute('autoplay', '');
      video.setAttribute('playsinline', '');
      
      const capture = document.createElement('button');
      capture.textContent = '📸 Capture & Deteksi';
      capture.style.cssText = 'margin-top:8px; background:#C9A84C; color:#0f0800; border:none; padding:10px 24px; border-radius:6px; font-weight:bold; font-size:14px; cursor:pointer; display:block; width:100%;';
      
      div.appendChild(label);
      div.appendChild(video);
      div.appendChild(capture);
      document.body.appendChild(div);
      
      try {
        const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: 'environment', width:{ideal:1280}, height:{ideal:720}}});
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => capture.onclick = resolve);
        
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      } catch(e) {
        div.innerHTML = '<p style="color:red;">❌ Akses kamera gagal: ' + e.message + '</p>';
        return null;
      }
    }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')
    if not data:
        return None
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

print("✅ Fungsi take_photo() siap.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import numpy as np
from IPython.display import display, HTML

# Pilih model: gunakan best.pt jika sudah training, fallback ke pretrained
if BEST_MODEL.exists():
    INFER_MODEL_PATH = str(BEST_MODEL)
    print(f"🎯 Menggunakan model fine-tuned: {INFER_MODEL_PATH}")
else:
    INFER_MODEL_PATH = 'yolo26n.pt'
    print(f"⚠️ best.pt tidak ditemukan — menggunakan pretrained: {INFER_MODEL_PATH}")

model_infer = YOLO(INFER_MODEL_PATH)

# Ambil gambar dari webcam
photo_file = take_photo('/content/captured_keris.jpg')

if photo_file:
    img = cv2.imread(photo_file)
    results = model_infer.predict(img, conf=0.25, iou=0.45, verbose=False)

    detections_found = []

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf  = float(box.conf[0].item())
            cls   = int(box.cls[0].item())
            # Gunakan nama dari CLASS_NAMES jika model fine-tuned, else dari r.names
            label = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else r.names.get(cls, 'keris_unknown')
            detections_found.append((x1, y1, x2, y2, conf, label))

            # Gambar bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (76, 168, 201), 3)
            text = f"{label} {conf:.0%}"
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(img, (x1, y1-th-10), (x1+tw+4, y1), (76, 168, 201), -1)
            cv2.putText(img, text, (x1+2, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (15, 8, 0), 2)

    # Tampilkan gambar hasil
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'YOLOv26 Kris Detection — {len(detections_found)} objek terdeteksi', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Tampilkan kartu metadata budaya untuk setiap deteksi
    if not detections_found:
        display(HTML("""
        <div style='background:#1a0e00;border:1px solid #C9A84C44;border-radius:8px;padding:16px;color:#9a8060;font-family:sans-serif;'>
            ℹ️ Tidak ada keris terdeteksi. Pastikan bilah keris terlihat jelas dan pencahayaan cukup.
        </div>"""))
    
    for i, (x1, y1, x2, y2, conf, label) in enumerate(detections_found):
        info = get_cultural_info(label)
        luk_display = f"Luk {info['luk']}" if info['luk'] is not None and info['luk'] > 0 else "Lurus (0 luk)"
        
        card_html = f"""
        <div style='background:linear-gradient(135deg,#1a0e00,#2d1a00);border:2px solid #C9A84C;
                    border-radius:10px;padding:18px;color:#f5e6c8;max-width:580px;
                    font-family:sans-serif;margin:10px 0;'>
          <h3 style='margin:0 0 12px;color:#C9A84C;border-bottom:1px solid #C9A84C33;
                     padding-bottom:8px;font-size:15px;'>🗡️ IDENTIFIKASI BUDAYA MADURA #{i+1}</h3>
          <div style='font-size:11px;color:#C9A84C;text-transform:uppercase;'>Deteksi YOLOv26</div>
          <div style='font-size:17px;font-weight:bold;margin:4px 0 14px;'>{label.replace('_',' ').title()} — {conf:.1%}</div>
          <div style='display:grid;grid-template-columns:1fr 1fr;gap:10px;font-size:13px;margin-bottom:12px;'>
            <div><b>⚔️ Perkiraan Dapur:</b> {info['dapur']}</div>
            <div><b>〰️ Jumlah Luk:</b> {luk_display}</div>
            <div><b>✨ Contoh Pamor:</b> {info['pamor']}</div>
            <div><b>🏛️ Tangguh:</b> {info['tangguh']}</div>
          </div>
          <div style='font-size:12px;background:rgba(201,168,76,0.08);padding:10px;
                      border-left:3px solid #C9A84C;border-radius:0 6px 6px 0;margin-bottom:10px;'>
            <b>📜 Makna Filosofis:</b><br/>{info['luk_makna']}
          </div>
          <div style='font-size:11px;color:#90caf9;'>🌐 <b>UNESCO:</b> {info['unesco']}</div>
        </div>"""
        display(HTML(card_html))
else:
    print("❌ Gagal mengambil gambar dari webcam.")

## 9. 🖼️ Inference pada Gambar Dataset (Tanpa Webcam)

Uji model pada beberapa gambar dari dataset yang sudah ada — berguna untuk validasi manual.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

# Ambil sample gambar dari val set
val_image_dir = IMAGES_DIR / 'val'
if not val_image_dir.exists() or not list(val_image_dir.glob('*.*')):
    val_image_dir = IMAGES_DIR / 'train'
    print("ℹ️ Menggunakan gambar dari train set (val set kosong)")

sample_images = random.sample(
    [f for f in val_image_dir.iterdir() if f.suffix.lower() in ('.jpg','.jpeg','.png')],
    k=min(6, len(list(val_image_dir.iterdir())))
)

print(f"🔍 Menjalankan inference pada {len(sample_images)} gambar sample...\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_images):
    img = cv2.imread(str(img_path))
    results = model_infer.predict(img, conf=0.25, verbose=False)
    
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0].item())
            cls  = int(box.cls[0].item())
            label = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else r.names.get(cls, '?')
            cv2.rectangle(img, (x1, y1), (x2, y2), (76, 168, 201), 3)
            cv2.putText(img, f"{label[:20]} {conf:.0%}", (x1, max(y1-8, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (201, 168, 76), 2)
    
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name[:40], fontsize=9)
    ax.axis('off')

for ax in axes[len(sample_images):]:
    ax.axis('off')

plt.suptitle('Sample Inference YOLOv26 — Dataset Keris Madura', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()